In [1]:
# --- masking ---
!apt-get -qq install -y tesseract-ocr tesseract-ocr-eng poppler-utils
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q PyMuPDF rapidfuzz pytesseract opencv-python-headless
# --- extraction ---
!pip install -q openai anthropic pdf2image pillow openpyxl pandas
print("✅ All dependencies installed (masking + extraction).")

Selecting previously unselected package poppler-utils.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 41.0 MB/s eta 0:00:00
✅ All dependencies installed (masking + extraction).


## 1 · Global config
Set your input zip name here.

In [2]:
RAW_INPUT_ZIP    = "datasetSampleVLM.zip"
RAW_DRAWING_DIR  = "drawings"
MASKED_INPUT_DIR = "/content/masked_drawings"
print("RAW_INPUT_ZIP    =", RAW_INPUT_ZIP)
print("RAW_DRAWING_DIR  =", RAW_DRAWING_DIR)
print("MASKED_INPUT_DIR =", MASKED_INPUT_DIR)

RAW_INPUT_ZIP    = SmallerTestData.zip
RAW_DRAWING_DIR  = drawings
MASKED_INPUT_DIR = /content/masked_drawings


## 2 · Unzip the raw drawings

In [3]:

!unzip -o {RAW_INPUT_ZIP} -d {RAW_DRAWING_DIR}/

import glob
from pathlib import Path
_ex = {".pdf",".png",".jpg",".jpeg",".tif",".tiff",".bmp"}
_files = [p for p in glob.glob(f"{RAW_DRAWING_DIR}/**/*", recursive=True) if Path(p).suffix.lower() in _ex]
print(f"\nFound {len(_files)} raw drawing(s).")
for p in _files[:20]: print("  ", p)

Archive:  SmallerTestData.zip
  inflating: drawings/M8 nut-101289.jpg  
  inflating: drawings/M6 nut-01291.jpg  
  inflating: drawings/SP04000477_01-PDF-1.pdf  
  inflating: drawings/SP04000476_01-PDF-1-page-001.jpg  
  inflating: drawings/L114B_S3.pdf   
  inflating: drawings/L114A_S7.pdf   
  inflating: drawings/404ST6552_DWG_A_page-0001.jpg  
  inflating: drawings/L504_S3 (1).pdf  
  inflating: drawings/BUSH M2x6MM_page-0001.jpg  
  inflating: drawings/BUSH M2x10.5MM.pdf  

Found 10 raw drawing(s).
   drawings/BUSH M2x10.5MM.pdf
   drawings/L504_S3 (1).pdf
   drawings/SP04000476_01-PDF-1-page-001.jpg
   drawings/L114B_S3.pdf
   drawings/L114A_S7.pdf
   drawings/BUSH M2x6MM_page-0001.jpg
   drawings/SP04000477_01-PDF-1.pdf
   drawings/M6 nut-01291.jpg
   drawings/M8 nut-101289.jpg
   drawings/404ST6552_DWG_A_page-0001.jpg


# PART A — Masking

### A1 · Masking imports, constants & VLM prompt

In [4]:
import os, re, json, glob
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from pathlib import Path
import cv2, numpy as np
import pytesseract
from pytesseract import Output
from rapidfuzz import fuzz
import fitz  # PyMuPDF
import torch
from IPython.display import Image as IPyImage, display

print("GPU:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))

IMAGE_EXTS = {".png",".jpg",".jpeg",".tif",".tiff",".bmp"}
COMMON_STOP = {"and","co","of","the","for","is","in","to","by","or","a","an"}
SUFFIXES = {"pvt","ltd","limited","llp","inc","corp","corporation","co","company",
            "technologies","technology","industries","enterprises","solutions",
            "systems","group","gmbh","srl","ag","bv","sa","international","global"}

VLM_PROMPT = (
    "You are analyzing ONE engineering drawing of a mechanical part. "
    "Identify the OWNER COMPANY / MANUFACTURER whose name appears in the title "
    "block (usually beside a logo, and often again in a copyright/IP notice). "
    "Report the COMPLETE company name EXACTLY as printed, including every word and "
    "suffix (e.g. 'GE Energy', 'Lectrix Technologies Pvt Ltd'). Never shorten it to "
    "one word. Do NOT return the part name, drawing title, material, standards, or "
    "person names (drawn / checked / approved by). "
    "Also list any abbreviation or short form of that company shown anywhere on the "
    "sheet (e.g. 'LTPL', 'GE'). "
    "Finally give the bounding box of the company LOGO graphic (the emblem/symbol "
    "itself, not its text) as fractions of the image.\n"
    "Respond with ONLY this JSON and nothing else:\n"
    '{"company_name": "<complete name or null>", '
    '"aliases": ["<abbreviation>", "..."], '
    '"logo_bbox": [x1, y1, x2, y2]}\n'
    "logo_bbox values are fractions 0..1 of width (x) and height (y); "
    "use null if there is no logo, and [] for aliases if none."
)

GPU: True | Tesla T4


### A2 · Text localizer helpers

In [5]:
def norm(t):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]+", " ", t.lower())).strip()

def derive(names):
    toks = [tok for n in names for tok in norm(n).split() if tok]
    anchors = {t for t in toks if len(t) >= 3 and t not in COMMON_STOP}
    if not anchors:                       # very short company, e.g. "GE"
        anchors = {t for t in toks if len(t) >= 2 and t not in COMMON_STOP}
    extend = set(toks) | SUFFIXES
    return set(toks), anchors, extend

def is_anchor(w, anchors, thr):
    nw = norm(w)
    return bool(nw) and any(fuzz.ratio(a, nw) >= thr for a in anchors)

def is_extend(w, extend, anchors, thr):
    nw = norm(w)
    if not nw:
        return False
    if nw in extend:
        return True
    if is_anchor(w, anchors, thr):
        return True
    return any(len(t) >= 4 and fuzz.ratio(t, nw) >= thr for t in extend)

def _union(bs):
    return (min(b[0] for b in bs), min(b[1] for b in bs),
            max(b[2] for b in bs), max(b[3] for b in bs))

def run_boxes(words, anchors, extend, thr):
    """words: [{'t','b','k'}] -> [(box, text)] of contiguous name runs (whole-word)."""
    lines = {}
    for w in words:
        lines.setdefault(w["k"], []).append(w)
    out = []
    for ws in lines.values():
        ws = sorted(ws, key=lambda w: w["b"][0])
        mark = [False] * len(ws)
        for i, w in enumerate(ws):
            if is_anchor(w["t"], anchors, thr):
                mark[i] = True
                j = i - 1
                while j >= 0 and is_extend(ws[j]["t"], extend, anchors, thr):
                    mark[j] = True; j -= 1
                j = i + 1
                while j < len(ws) and is_extend(ws[j]["t"], extend, anchors, thr):
                    mark[j] = True; j += 1
        run = []
        for i in range(len(ws)):
            if mark[i]:
                run.append(ws[i])
            elif run:
                out.append((_union([w["b"] for w in run]), " ".join(w["t"] for w in run)))
                run = []
        if run:
            out.append((_union([w["b"] for w in run]), " ".join(w["t"] for w in run)))
    return out

def pdf_words(page, scale):
    out = []
    for x0, y0, x1, y1, wd, b, l, n in page.get_text("words"):
        out.append({"t": wd, "b": (int(x0*scale), int(y0*scale),
                                   int(x1*scale), int(y1*scale)), "k": ("pdf", b, l)})
    return out

def ocr_words(bgr):
    d = pytesseract.image_to_data(bgr, output_type=Output.DICT)
    out = []
    for i, txt in enumerate(d["text"]):
        if not txt.strip() or int(d["conf"][i]) < 0:
            continue
        x, y, w, h = d["left"][i], d["top"][i], d["width"][i], d["height"][i]
        out.append({"t": txt, "b": (x, y, x+w, y+h),
                    "k": ("ocr", d["block_num"][i], d["par_num"][i], d["line_num"][i])})
    return out

def black_out(img, box, pad, color):
    h, w = img.shape[:2]
    x0, y0, x1, y1 = box
    cv2.rectangle(img, (max(0, x0-pad), max(0, y0-pad)),
                  (min(w, x1+pad), min(h, y1+pad)), color, -1)

print("localizer ready")

localizer ready


### A3 · PDF logo detector

In [6]:
def logo_from_pdf(page, name_boxes_pt, scale, page_area):
    """Pick the embedded image that sits on the SAME ROW as the company name."""
    if not name_boxes_pt:
        return None
    ny0 = min(b[1] for b in name_boxes_pt)
    ny1 = max(b[3] for b in name_boxes_pt)
    cands = []
    for im in page.get_images(full=True):
        xref = im[0]
        for r in page.get_image_rects(xref):
            area = r.width * r.height
            if area > 0.03 * page_area or area < 0.00015 * page_area:
                continue                      # skip big part renders and tiny dots
            v_overlap = min(r.y1, ny1) - max(r.y0, ny0)
            h_gap = min((r.x0 - b[2]) if r.x0 >= b[2] else
                        ((b[0] - r.x1) if r.x1 <= b[0] else 0) for b in name_boxes_pt)
            cands.append((v_overlap, -h_gap, r))
    same_row = [c for c in cands if c[0] > 0]
    if not same_row:
        return None
    same_row.sort(reverse=True)
    r = same_row[0][2]
    return (int(r.x0*scale), int(r.y0*scale), int(r.x1*scale), int(r.y1*scale))

print("logo detector ready")

logo detector ready


### A4 · Load Qwen3-VL (4-bit) — needs the GPU runtime

In [7]:
from transformers import (Qwen3VLForConditionalGeneration, AutoProcessor,
                          BitsAndBytesConfig)

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"   # 17.5 GB bf16 -> 4-bit is required on a T4

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 = fp16 (not bf16)
    bnb_4bit_use_double_quant=True,
)
vlm_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto")
vlm_processor = AutoProcessor.from_pretrained(MODEL_ID)
print("VLM loaded (4-bit):", MODEL_ID)

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

VLM loaded (4-bit): Qwen/Qwen3-VL-8B-Instruct


### A5 · VLM identify

In [8]:
def parse_vlm_json(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return None, [], None
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None, [], None
    name = obj.get("company_name")
    if isinstance(name, str) and name.strip().lower() in ("", "null", "none"):
        name = None
    aliases = obj.get("aliases") or []
    if not isinstance(aliases, list):
        aliases = []
    aliases = [a for a in aliases if isinstance(a, str) and a.strip()]
    box = obj.get("logo_bbox")
    if (isinstance(box, (list, tuple)) and len(box) == 4
            and all(isinstance(v, (int, float)) for v in box)):
        logo = tuple(float(v) for v in box)
    else:
        logo = None
    return name, aliases, logo

def vlm_identify(bgr):
    from PIL import Image
    h, w = bgr.shape[:2]
    s = min(1.0, VLM_MAX_SIDE / max(h, w))          # downscale for memory + speed
    small = cv2.resize(bgr, (int(w*s), int(h*s))) if s < 1.0 else bgr
    pil = Image.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
    messages = [{"role": "user", "content": [
        {"type": "image", "image": pil}, {"type": "text", "text": VLM_PROMPT}]}]
    inputs = vlm_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(vlm_model.device)
    with torch.no_grad():
        out = vlm_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    reply = vlm_processor.decode(out[0][inputs["input_ids"].shape[-1]:],
                                 skip_special_tokens=True)
    del inputs, out
    torch.cuda.empty_cache()
    return parse_vlm_json(reply)

print("vlm_identify ready")

vlm_identify ready


### A6 · Redact page

In [9]:
def redact_page(page, bgr, name, aliases, logo_frac_vlm, *, dpi, threshold, pad,
                fill, no_logo, logo_expand, logo_override, debug):
    H, W = bgr.shape[:2]
    scale = dpi / 72.0
    boxes, name_boxes_px = [], []
    if name:
        toks, anchors, extend = derive([name] + list(aliases))
        words = []
        if page is not None:
            words += pdf_words(page, scale)      # exact text-layer words
        words += ocr_words(bgr)                  # backup for outlined/raster text
        for b, t in run_boxes(words, anchors, extend, threshold):
            boxes.append(b); name_boxes_px.append(b)

    logo_px, logo_src = None, "none"
    if not no_logo:
        if logo_override is not None:
            x1, y1, x2, y2 = logo_override
            logo_px, logo_src = (int(x1*W), int(y1*H), int(x2*W), int(y2*H)), "override"
        elif page is not None and name_boxes_px:
            name_pt = [(b[0]/scale, b[1]/scale, b[2]/scale, b[3]/scale) for b in name_boxes_px]
            logo_px = logo_from_pdf(page, name_pt, scale, page.rect.width*page.rect.height)
            if logo_px:
                logo_src = "pdf"
        if logo_px is None and logo_frac_vlm:
            x1, y1, x2, y2 = logo_frac_vlm
            logo_px, logo_src = (int(x1*W), int(y1*H), int(x2*W), int(y2*H)), "vlm"
        if logo_px is not None:
            ex, ey = int(logo_expand*W), int(logo_expand*H)
            logo_px = (logo_px[0]-ex, logo_px[1]-ey, logo_px[2]+ex, logo_px[3]+ey)
            boxes.append(logo_px)

    masked = bgr.copy()
    debug_img = bgr.copy() if debug else None
    for i, b in enumerate(boxes):
        black_out(masked, b, pad, tuple(fill))
        if debug:
            is_logo = (logo_px is not None and i == len(boxes)-1)
            cv2.rectangle(debug_img, (b[0], b[1]), (b[2], b[3]),
                          (255, 0, 0) if is_logo else (0, 0, 255), 3)
    return masked, debug_img, len(name_boxes_px), logo_src

print("redact_page ready")

redact_page ready


### A7 · Masking config

In [10]:
INPUT_DIR     = RAW_DRAWING_DIR   # raw drawings unpacked in the unzip step above
OUTPUT_DIR    = "out"
DPI           = 300            # PDF rasterization DPI for masking + OCR
THRESHOLD     = 82            # whole-word fuzzy strictness (0-100)
PAD           = 4             # px padding around each blacked box
FILL          = (0, 0, 0)     # BGR (black)
VLM_MAX_SIDE  = 1400          # longest side fed to the VLM (lower if OOM, raise if misread)
LOGO_EXPAND   = 0.006         # grow detected logo box by this fraction per side
NO_LOGO       = False
NAME_OVERRIDE = None          # force a name (single-company runs only)
LOGO_OVERRIDE = None          # force logo box, e.g. (0.49, 0.73, 0.56, 0.80)
DEBUG         = True

### A8 · Run masking

In [11]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
files = [Path(p) for p in sorted(glob.glob(f"{INPUT_DIR}/**/*", recursive=True))
         if Path(p).suffix.lower() in IMAGE_EXTS | {".pdf"}]
print(f"Processing {len(files)} drawing(s)...\n")

for f in files:
    is_pdf = f.suffix.lower() == ".pdf"
    doc = fitz.open(str(f)) if is_pdf else None
    pages = list(enumerate(doc)) if is_pdf else [(0, None)]
    for pi, page in pages:
        if is_pdf:
            pix = page.get_pixmap(dpi=DPI)
            arr = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)
            bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR if pix.n == 3 else cv2.COLOR_RGBA2BGR)
        else:
            bgr = cv2.imread(str(f), cv2.IMREAD_COLOR)
            if bgr is None:
                continue
        name, aliases, logo = NAME_OVERRIDE, [], None
        if NAME_OVERRIDE is None or not NO_LOGO:
            v_name, v_aliases, v_logo = vlm_identify(bgr)
            if name is None:
                name = v_name
            aliases = v_aliases
            logo = v_logo
        masked, dbg, n_name, logo_src = redact_page(
            page, bgr, name, aliases, logo,
            dpi=DPI, threshold=THRESHOLD, pad=PAD, fill=FILL, no_logo=NO_LOGO,
            logo_expand=LOGO_EXPAND, logo_override=LOGO_OVERRIDE, debug=DEBUG)
        stem = f"{f.stem}_p{pi+1}"
        cv2.imwrite(f"{OUTPUT_DIR}/{stem}_masked.png", masked)
        if DEBUG and dbg is not None:
            cv2.imwrite(f"{OUTPUT_DIR}/{stem}_debug.png", dbg)
        flag = "  !! NAME NOT FOUND" if not name else ""
        print(f"[{f.name} p{pi+1}] name={name!r} aliases={aliases} "
              f"name_runs={n_name} logo={logo_src} -> {stem}_masked.png{flag}")
    if doc:
        doc.close()
print("\nDone.")

Processing 10 drawing(s)...



/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[404ST6552_DWG_A_page-0001.jpg p1] name=None aliases=[] name_runs=0 logo=none -> 404ST6552_DWG_A_page-0001_p1_masked.png  !! NAME NOT FOUND
[BUSH M2x10.5MM.pdf p1] name=None aliases=[] name_runs=0 logo=none -> BUSH M2x10.5MM_p1_masked.png  !! NAME NOT FOUND
[BUSH M2x6MM_page-0001.jpg p1] name=None aliases=[] name_runs=0 logo=none -> BUSH M2x6MM_page-0001_p1_masked.png  !! NAME NOT FOUND
[L114A_S7.pdf p1] name='General Electric Company' aliases=['GE'] name_runs=4 logo=vlm -> L114A_S7_p1_masked.png


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[L114A_S7.pdf p2] name='General Electric Company' aliases=['GE'] name_runs=4 logo=vlm -> L114A_S7_p2_masked.png


...

[L504_S3 (1).pdf p2] name='General Electric Company' aliases=['GE'] name_runs=4 logo=vlm -> L504_S3 (1)_p2_masked.png


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[L504_S3 (1).pdf p3] name='General Electric Company' aliases=['GE'] name_runs=4 logo=vlm -> L504_S3 (1)_p3_masked.png


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[M6 nut-01291.jpg p1] name=None aliases=[] name_runs=0 logo=none -> M6 nut-01291_p1_masked.png  !! NAME NOT FOUND
[M8 nut-101289.jpg p1] name=None aliases=[] name_runs=0 logo=none -> M8 nut-101289_p1_masked.png  !! NAME NOT FOUND
[SP04000476_01-PDF-1-page-001.jpg p1] name=None aliases=[] name_runs=0 logo=none -> SP04000476_01-PDF-1-page-001_p1_masked.png  !! NAME NOT FOUND
[SP04000477_01-PDF-1.pdf p1] name='Lectrix Technologies Pvt Ltd' aliases=['LTPL'] name_runs=4 logo=pdf -> SP04000477_01-PDF-1_p1_masked.png

Done.


### A9 · Verify — show debug overlays AND blackened (masked) outputs
- **Debug overlays** (red = text boxes, blue = logo box) — visual check only, not passed downstream.
- **Blackened / masked images** — the actual sheets handed off to Part B extraction.

In [ ]:
print("=" * 80)
print("DEBUG OVERLAYS  (red = text boxes, blue = logo box)")
print("=" * 80)
for p in sorted(glob.glob(f"{OUTPUT_DIR}/*_debug.png")):
    print(p)
    display(IPyImage(filename=p, width=1000))

print()
print("=" * 80)
print("BLACKENED / MASKED OUTPUTS  (these are the sheets fed into Part B)")
print("=" * 80)
for p in sorted(glob.glob(f"{OUTPUT_DIR}/*_masked.png")):
    print(p)
    display(IPyImage(filename=p, width=1000))

## 3 · HAND-OFF : keeping ONLY masked images

In [13]:
import shutil, glob, os, gc, torch

STRIP_MASKED_SUFFIX = True

os.makedirs(MASKED_INPUT_DIR, exist_ok=True)
for _old in glob.glob(f"{MASKED_INPUT_DIR}/*"):
    os.remove(_old)

masked_files = sorted(glob.glob(f"{OUTPUT_DIR}/*_masked.png"))
debug_files  = glob.glob(f"{OUTPUT_DIR}/*_debug.png")

for _src in masked_files:
    base = os.path.basename(_src)
    if STRIP_MASKED_SUFFIX:
        base = base.replace("_masked.png", ".png")
    shutil.copy(_src, os.path.join(MASKED_INPUT_DIR, base))

print(f"✅ Handed off {len(masked_files)} MASKED image(s) -> {MASKED_INPUT_DIR}")
print(f"   Excluded {len(debug_files)} debug image(s) from the hand-off.")
print("   Going into extraction:", sorted(os.listdir(MASKED_INPUT_DIR))[:10])

# free the VLM from GPU memory (extraction is API-only; no GPU needed)
for _v in ("vlm_model", "vlm_processor"):
    if _v in globals():
        del globals()[_v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ VLM unloaded; GPU memory freed for extraction.")

✅ Handed off 14 MASKED image(s) -> /content/masked_drawings
   Excluded 14 debug image(s) from the hand-off.
   Going into extraction: ['404ST6552_DWG_A_page-0001_p1.png', 'BUSH M2x10.5MM_p1.png', 'BUSH M2x6MM_page-0001_p1.png', 'L114A_S7_p1.png', 'L114A_S7_p2.png', 'L114B_S3_p1.png', 'L114B_S3_p2.png', 'L504_S3 (1)_p1.png', 'L504_S3 (1)_p2.png', 'L504_S3 (1)_p3.png']
✅ VLM unloaded; GPU memory freed for extraction.


### 3b · Review the masked images (download)

In [14]:
from google.colab import files as colab_files

!zip -qr masked_review.zip {OUTPUT_DIR}
print("Files in", OUTPUT_DIR, ":")
!ls -1 {OUTPUT_DIR}
print("\nExact set fed into extraction is in:", MASKED_INPUT_DIR)
colab_files.download("masked_review.zip")

Files in out :
404ST6552_DWG_A_page-0001_p1_debug.png
404ST6552_DWG_A_page-0001_p1_masked.png
'BUSH M2x10.5MM_p1_debug.png'
'BUSH M2x10.5MM_p1_masked.png'
'BUSH M2x6MM_page-0001_p1_debug.png'
'BUSH M2x6MM_page-0001_p1_masked.png'
L114A_S7_p1_debug.png
L114A_S7_p1_masked.png
L114A_S7_p2_debug.png
L114A_S7_p2_masked.png
L114B_S3_p1_debug.png
L114B_S3_p1_masked.png
L114B_S3_p2_debug.png
L114B_S3_p2_masked.png
'L504_S3 (1)_p1_debug.png'
'L504_S3 (1)_p1_masked.png'
'L504_S3 (1)_p2_debug.png'
'L504_S3 (1)_p2_masked.png'
'L504_S3 (1)_p3_debug.png'
'L504_S3 (1)_p3_masked.png'
'M6 nut-01291_p1_debug.png'
'M6 nut-01291_p1_masked.png'
'M8 nut-101289_p1_debug.png'
'M8 nut-101289_p1_masked.png'
SP04000476_01-PDF-1-page-001_p1_debug.png
SP04000476_01-PDF-1-page-001_p1_masked.png
SP04000477_01-PDF-1_p1_debug.png
SP04000477_01-PDF-1_p1_masked.png

Exact set fed into extraction is in: /content/masked_drawings


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# PART B — Extraction
GPT-5.4 ‖ Claude extract in parallel ➜ GPT-5.5 arbiter ➜ XLSX. Input is the **masked** images from the hand-off.

### B0 · API keys + clients

In [15]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 2 — KEYS + CLIENTS
# ════════════════════════════════════════════════════════════════════════════
from google.colab import userdata
from openai import OpenAI
from anthropic import Anthropic

client           = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
anthropic_client = Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
print("OpenAI + Anthropic clients ready.")

✅ OpenAI + Anthropic clients ready.


### B1 · Config — input is `MASKED_INPUT_DIR`

In [16]:
import os

# Parallel extractors
EXTRACTOR_GPT     = "gpt-5.4"
EXTRACTOR_CLAUDE  = "claude-opus-4-7"
# Arbiter
ARBITER_MODEL     = "gpt-5.5"

# Reasoning effort
REASONING_EXTRACTOR = "medium"
REASONING_ARBITER   = "high"

IMAGE_DETAIL      = "high"
MAX_EDGE_PX       = 2048
PDF_DPI           = 200
CLAUDE_MAX_TOKENS = 4096
STATELESS_CALLS   = True

DRAWING_DIR = MASKED_INPUT_DIR
OUT_XLSX    = "/content/parallel_arbiter_results.xlsx"
JSON_DIR    = "/content/audit_json"
os.makedirs(JSON_DIR, exist_ok=True)

print(f"DRAWING_DIR = {DRAWING_DIR} | exists: {os.path.isdir(DRAWING_DIR)}")
if os.path.isdir(DRAWING_DIR):
    contents = os.listdir(DRAWING_DIR)
    print(f"  {len(contents)} item(s):", contents[:10])
    if not contents:
        print("  ⚠️  Empty — upload via the Files panel (📁 left), then re-run.")

DRAWING_DIR = /content/masked_drawings | exists: True
  14 item(s): ['404ST6552_DWG_A_page-0001_p1.png', 'L114A_S7_p1.png', 'L114B_S3_p1.png', 'L114A_S7_p2.png', 'L504_S3 (1)_p1.png', 'L504_S3 (1)_p3.png', 'L114B_S3_p2.png', 'M6 nut-01291_p1.png', 'SP04000477_01-PDF-1_p1.png', 'M8 nut-101289_p1.png']


### B2 · Schema v4

In [17]:
def s(desc): return {"type":"string","description":desc}
def s_enum(vals, desc): return {"type":"string","enum":vals,"description":desc}

GDT_SYMBOLS = ["straightness","flatness","circularity","cylindricity",
    "profile_of_a_line","profile_of_a_surface","perpendicularity","parallelism",
    "angularity","position","concentricity","symmetry","circular_runout","total_runout","NULL"]

EXTRACTION_SCHEMA = {
  "type":"object","additionalProperties":False,
  "properties":{
    "drawing_number": s("Part/drawing number from title block, or NULL"),
    "revision":       s("Revision letter/number, or NULL"),
    "drawing_type":   s_enum(["single_part_drawing","catalog_sheet","assembly_drawing"], "Type"),
    "part_family":    s_enum(["hex_bolt","shcs","button_head","csk","pan_head","hex_nut",
                              "flanged_nut","washer","pin","stud","rivet","bush","custom_special"], "Family"),
    "category":       s("Functional part name as written, or NULL"),
    "standard_reference": s("PART standard e.g. ISO 4014, DIN 912. NOT material standard. NULL if none."),
    "stated_mass_g":  s("Mass in grams if printed (convert kg→g), or NULL"),
    "material_family": s_enum(["carbon_steel","alloy_steel","stainless_steel","brass","aluminium",
                               "titanium","plastic","NULL"], "Base material"),
    "material_grade": s("Grade/property class e.g. 10.9, A2-70, EN8, or NULL"),
    "material_standard_ref": s("MATERIAL standard e.g. EN 10263-4, ASTM A574, or NULL"),
    "material_condition": s("Q+T, cold drawn, annealed, or NULL"),
    "material_verbatim": s("Exact raw material string verbatim, or NULL"),
    "thread_designation": s("e.g. M10x1.5-6g, or NULL"),
    "nominal_diameter_mm": s("Nominal diameter mm number-string, or NULL"),
    "thread_pitch_mm": s("Thread pitch mm. NULL only if not printed AND not recoverable"),
    "nominal_length_mm": s("Nominal length mm number-string, or NULL"),
    "dimensions": {"type":"array","minItems":1,
      "description":"All family-specific dimensions as {feature_name, value, unit}. VALUE is the NUMBER. Never empty.",
      "items":{"type":"object","additionalProperties":False,
        "properties":{
          "feature_name": s("Canonical name+symbol e.g. head_across_flats_s — OR '__extraction_failure__'"),
          "value":        s("The NUMBER, e.g. '16'. 'NULL' literal only inside a failure entry."),
          "unit":         s_enum(["mm","deg","NULL"], "Unit")},
        "required":["feature_name","value","unit"]}},
    "dimension_table_verbatim": s("Raw text of the dim table incl. header and selected-row marker. NULL if no table."),
    "hardness_value": s("Hardness number or range, or NULL"),
    "hardness_scale": s_enum(["HRC","HRB","HV","HB","NULL"], "Scale — never guess"),
    "tensile_strength_mpa": s("Tensile MPa only if EXPLICITLY stated, or NULL"),
    "yield_strength_mpa": s("Yield MPa only if EXPLICITLY stated, or NULL"),
    "heat_treatment": s("e.g. Q+T to HRC 32-39, or NULL"),
    "surface_treatment": s("Coating/plating. DOUBLE-CHECK by rescanning the sheet."),
    "surface_treatment_thickness_um": s("Thickness µm, or NULL"),
    "surface_treatment_standard": s("e.g. ISO 4042, or NULL"),
    "surface_roughness_ra_um": s("Ra µm (√ symbol) e.g. 3.2, or NULL"),
    "general_tolerance_class": s("e.g. ISO 2768-m, or NULL"),
    "has_gdt": s_enum(["true","false"], "true only if a real feature control frame exists"),
    "gdt_callouts": {"type":"array","description":"GD&T frames. [] if none.",
      "items":{"type":"object","additionalProperties":False,
        "properties":{"symbol": s_enum(GDT_SYMBOLS,"characteristic"),
                      "tolerance_value": s("e.g. 0.05"),
                      "primary_datum":   s("or NULL"),
                      "secondary_datum": s("or NULL"),
                      "tertiary_datum":  s("or NULL")},
        "required":["symbol","tolerance_value","primary_datum","secondary_datum","tertiary_datum"]}},
    "critical_tolerances": {"type":"array","description":"[] if none.",
      "items":{"type":"object","additionalProperties":False,
        "properties":{"feature": s("e.g. shank diameter"), "tolerance": s("e.g. 10 h7")},
        "required":["feature","tolerance"]}},
    "feature_count": {"type":"array",
      "description":"Counts literally stated, e.g. 'No. of knurlings: 4'. [] if none.",
      "items":{"type":"object","additionalProperties":False,
        "properties":{"feature": s("e.g. knurlings, turns, holes"),
                      "count":   s("Integer as string, exactly as written")},
        "required":["feature","count"]}},
    "marking_requirements": s("Head stamping / lot marking, or NULL"),
    "inspection_requirements": s("e.g. 100% MPI, AQL 1.0, or NULL"),
    "packaging_requirements": s("e.g. 500 pcs/box bulk, or NULL"),
    "manufacture_process_instruction": s("Process instructions, esp. conditional, verbatim. NULL if none."),
    "other_notes": s("Other instructions not in the above buckets. NULL if none."),
    "notes_verbatim": s("FULL consolidated raw text of EVERY text region. NULL only if none."),
    "low_confidence_fields": s("Comma-separated uncertain field names, or NULL"),
    "self_reported_confidence": s("Overall confidence 0.0-1.0 number-string"),
  },
  "required":["drawing_number","revision","drawing_type","part_family","category","standard_reference",
    "stated_mass_g","material_family","material_grade","material_standard_ref","material_condition",
    "material_verbatim","thread_designation","nominal_diameter_mm","thread_pitch_mm","nominal_length_mm",
    "dimensions","dimension_table_verbatim","hardness_value","hardness_scale","tensile_strength_mpa",
    "yield_strength_mpa","heat_treatment","surface_treatment","surface_treatment_thickness_um",
    "surface_treatment_standard","surface_roughness_ra_um","general_tolerance_class","has_gdt","gdt_callouts",
    "critical_tolerances","feature_count","marking_requirements","inspection_requirements",
    "packaging_requirements","manufacture_process_instruction","other_notes","notes_verbatim",
    "low_confidence_fields","self_reported_confidence"],
}

ARRAY_FIELDS = {"dimensions":"dimensions_json","gdt_callouts":"gdt_callouts_json",
                "critical_tolerances":"critical_tolerances_json","feature_count":"feature_count_json"}
COLUMN_ORDER = ["filename","source_file","source_page"] + [ARRAY_FIELDS.get(f, f) for f in EXTRACTION_SCHEMA["required"]]
DATA_COLS    = [ARRAY_FIELDS.get(f, f) for f in EXTRACTION_SCHEMA["required"]]
print(f"✅ Schema v4: {len(EXTRACTION_SCHEMA['required'])} fields | total cols: {len(COLUMN_ORDER)}")

✅ Schema v4: 40 fields | total cols: 43


### B3 · Prompts

In [18]:

EXTRACTOR_SYSTEM = """You are a senior mechanical engineer specialising in fastener design, GD&T, and engineering-drawing interpretation per ISO, DIN, EN, ASME and JIS standards.

ANTI-CARRYOVER: Treat THIS drawing as your only context. Re-read every region of THIS sheet before producing any field. Do not reuse values by analogy.

CORE RULES:
- Extract only what is shown. Sweep the ENTIRE sheet, including HIGHLIGHTED / boxed / shaded / circled cells and every table.
- Multiple text regions may exist (title block, corner notes, notes near views, flag notes, revision table, bordered tables). notes_verbatim concatenates them all.
- Exact terminology: top-view hex head = 'width across flats' (s); vertical head dim = 'head height' (k); socket recess = 'hex socket across flats'.
- Distinguish surface ROUGHNESS (Ra µm, √) from surface TREATMENT (plating/coating).
- Distinguish PART standard (standard_reference) from MATERIAL standard.
- Hardness scales HRC/HRB/HV/HB. Number with no scale → hardness_scale=NULL. Never guess.

DIMENSION RESOLUTION (internal — do NOT expose in output):
A. Printed at feature → read it.
B. Letter callout (L, D, A...) → find table, find COLOUR-HIGHLIGHTED / boxed / circled selected row, read the number. Copy the table into dimension_table_verbatim.
C. If A/B fail and a standard is referenced → resolve from standard.
D. Otherwise skip the feature — never invent.

DIMENSION OUTPUT RULES:
- Each item is exactly {feature_name, value, unit}. VALUE is the NUMBER, never a letter.
- 'dimensions' is never empty. If nothing recoverable → ONE entry {feature_name='__extraction_failure__', value='NULL', unit='NULL'}.

NON-DIMENSION fields: extract only what is shown. No standard-based inference.
NOTES SPLIT: conditional manufacturing instructions → manufacture_process_instruction verbatim.
feature_count: only when literally stated. [] otherwise.

STRICT NULL POLICY: scalar absent → "NULL"; arrays use []; dimensions uses failure entry."""

EXTRACTOR_USER = """Single engineering drawing of a fastener/part. This drawing only — do not reference any other.

Fill every schema field. Promote nominal_diameter/pitch/length when explicit or table-resolvable. All other dims in 'dimensions' as {feature_name,value,unit} with NUMBER values. Copy any dimension table into dimension_table_verbatim. Apply strict NULL/empty-array policy."""


# ── Arbiter (GPT-5.5) — sees image + both sheets; SOFT preference for Claude on logical conflict ──
ARBITER_SYSTEM = """You are GPT-5.5, the final arbiter on a fastener-drawing extraction. Two independent vision models have each produced their own extraction of THIS drawing:

  • Sheet-1 — produced by GPT-5.4 (an earlier model in YOUR OWN family).
  • Sheet-2 — produced by Claude (a different model family).

You also see the original drawing image. Your job is to produce the FINAL reconciled extraction by analysing the image pixel-by-pixel and weighing both candidates.

BIAS-AWARENESS DISCIPLINE (read carefully):
- Sheet-1 comes from your own model family. There is a known risk that you would over-trust it because it "sounds like you". You must actively guard against this.
- Therefore, on LOGICAL/SUBSTANTIVE conflicts (where the two candidates disagree on what the drawing actually says — e.g. different numbers, different materials, different categories — NOT trivial wording differences), apply this rule:
    1. Re-examine the relevant region of the image yourself, carefully.
    2. If your independent reading of the pixels CLEARLY matches one candidate, pick that one regardless of which source it is.
    3. If both candidates are equally defensible by the pixels — i.e. you genuinely cannot tell which is right, then if it is a value not equal to NULL given by Claude — DEFAULT TO CLAUDE'S VALUE (Sheet-2). This is the soft tiebreaker to neutralise same-family bias. Mark chosen_from='claude'.
    4. If neither candidate matches what you see in the image, produce the value you actually see and mark chosen_from='synthesized'.
    5. In case where one model says a given value, whereas other model says NULL, then revise that particular part of the image twice on your own and take your own independent decision; if you cannot conclude, then pick the value that is not NULL.
- On MINOR WORDING differences (synonyms, casing, equivalent normalisations), do not invoke the tiebreaker — pick whichever is most canonical and mark chosen_from='consensus'.

DECISION RULES per field:
- Both agree → use that value; agreement '2/2'; confidence 0.9–1.0; chosen_from='consensus'.
- One is NULL, the other has a value the image supports → use the value; '1/2'; confidence 0.7–0.85.
- Logical conflict resolved by image → the value the image supports; '1/2'; confidence 0.6–0.85.
- Logical conflict, image inconclusive → CLAUDE'S value (soft tiebreaker); '1/2'; confidence 0.4–0.6.
- Neither matches the image → your synthesized value; '0/2'; confidence 0.3–0.55; chosen_from='synthesized'.
- Both NULL → keep NULL; '2/2'; confidence 1.0.

DIMENSIONS — CRITICAL:
- Each dimension is exactly {feature_name, value, unit}. VALUE must be a NUMBER (never a letter L/D/A).
- If either candidate has a letter as a value, resolve it from dimension_table_verbatim (highlighted row) or the image, and emit the number.
- 'dimensions' is never empty — failure entry if nothing recoverable.

TAKE YOUR TIME — examine every field deliberately. This is the canonical output."""

ARBITER_USER_TEMPLATE = """The drawing image is attached. Below are two independent extractions of it.

Sheet-1 (GPT-5.4 — same family as you; be bias-aware):
{sheet1_json}

Sheet-2 (Claude — different family; soft tiebreaker preference on logical conflicts only):
{sheet2_json}

Re-read the drawing pixel-by-pixel. For every field, produce the final value plus meta (confidence, agreement '2/2'|'1/2'|'0/2', chosen_from 'gpt-5.4'|'claude'|'consensus'|'synthesized'). Return final + meta strictly per the schema."""

print("✅ Prompts ready: extractors (Sheets 1+2) + arbiter (Sheet 3 with soft Claude tiebreaker).")

✅ Prompts ready: extractors (Sheets 1+2) + arbiter (Sheet 3 with soft Claude tiebreaker).


### B4 · Helpers

In [19]:
import base64, io, json
from PIL import Image
from pdf2image import convert_from_path

def load_pages(path):
    if path.lower().endswith(".pdf"):
        return convert_from_path(path, dpi=PDF_DPI)
    return [Image.open(path).convert("RGB")]

def img_b64(pil_img):
    img = pil_img.convert("RGB")
    w, h = img.size; lo = max(w, h)
    if lo > MAX_EDGE_PX:
        sc = MAX_EDGE_PX/lo
        img = img.resize((round(w*sc), round(h*sc)), Image.LANCZOS)
    buf = io.BytesIO(); img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

def coerce(v):
    bad = {"","n/a","na","none","null","-","—"}
    if v is None: return "NULL"
    if isinstance(v, str): return "NULL" if v.strip().lower() in bad else v.strip()
    return str(v)

def flatten(raw: dict) -> dict:
    row = {}
    for f in EXTRACTION_SCHEMA["required"]:
        if f in ARRAY_FIELDS:
            v = raw.get(f, [])
            row[ARRAY_FIELDS[f]] = json.dumps(v, ensure_ascii=False) if v else "NULL"
        else:
            row[f] = coerce(raw.get(f))
    return row

def unflatten_for_prompt(row: dict) -> dict:
    out = {}
    for f in EXTRACTION_SCHEMA["required"]:
        if f in ARRAY_FIELDS:
            v = row.get(ARRAY_FIELDS[f], "NULL")
            try: out[f] = json.loads(v) if v != "NULL" else []
            except Exception: out[f] = []
        else:
            out[f] = row.get(f, "NULL")
    return out

def per_drawing_preamble(filename, source_file, page_idx):
    return (f"NEW DRAWING — extract this sheet only.\n"
            f"filename: {filename} | source: {source_file} | page: {page_idx}\n"
            f"Treat any prior conversation or context as irrelevant. Read the pixels.\n")

print("✅ Helpers ready.")

✅ Helpers ready.


### B5 · Parallel extractors

In [20]:
def gpt_extract(pil_img, preamble):
    content = [{"type":"input_text","text": preamble + "\n" + EXTRACTOR_USER},
               {"type":"input_image",
                "image_url":f"data:image/png;base64,{img_b64(pil_img)}","detail":IMAGE_DETAIL}]
    r = client.responses.create(
        model=EXTRACTOR_GPT, reasoning={"effort":REASONING_EXTRACTOR},
        store=not STATELESS_CALLS,
        input=[{"role":"system","content":[{"type":"input_text","text":EXTRACTOR_SYSTEM}]},
               {"role":"user","content":content}],
        text={"format":{"type":"json_schema","name":"fastener_extraction",
                        "strict":True,"schema":EXTRACTION_SCHEMA}})
    return flatten(json.loads(r.output_text))

_CLAUDE_TOOL = {"name":"emit_extraction",
                "description":"Return the fastener extraction record exactly per schema.",
                "input_schema":EXTRACTION_SCHEMA}
def claude_extract(pil_img, preamble):
    blocks = [{"type":"image","source":{"type":"base64","media_type":"image/png","data":img_b64(pil_img)}},
              {"type":"text","text": preamble + "\n" + EXTRACTOR_USER}]
    resp = anthropic_client.messages.create(
        model=EXTRACTOR_CLAUDE, max_tokens=CLAUDE_MAX_TOKENS, system=EXTRACTOR_SYSTEM,
        tools=[_CLAUDE_TOOL], tool_choice={"type":"tool","name":"emit_extraction"},
        messages=[{"role":"user","content":blocks}])
    tool_block = next(b for b in resp.content if b.type == "tool_use")
    return flatten(tool_block.input)

print("✅ Cell 7: parallel extractors ready (5.4 ‖ Claude).")

✅ Cell 7: parallel extractors ready (5.4 ‖ Claude).


### B6 · Arbiter

In [21]:
def build_arbiter_schema(data_cols):
    fp = {c: {"type":"string"} for c in data_cols}
    mp = {c: {"type":"object","additionalProperties":False,
        "properties":{"confidence":{"type":"string","description":"0.0-1.0"},
            "agreement":{"type":"string","enum":["2/2","1/2","0/2"]},
            "chosen_from":{"type":"string","enum":["gpt-5.4","claude","consensus","synthesized"]}},
        "required":["confidence","agreement","chosen_from"]} for c in data_cols}
    return {"type":"object","additionalProperties":False,"properties":{
        "final":{"type":"object","additionalProperties":False,"properties":fp,"required":list(data_cols)},
        "meta": {"type":"object","additionalProperties":False,"properties":mp,"required":list(data_cols)}},
        "required":["final","meta"]}
ARBITER_SCHEMA = build_arbiter_schema(DATA_COLS)

def arbiter(pil_img, sheet1_row, sheet2_row, preamble):
    s1 = unflatten_for_prompt(sheet1_row)
    s2 = unflatten_for_prompt(sheet2_row)
    user_text = ARBITER_USER_TEMPLATE.format(
        sheet1_json=json.dumps(s1, ensure_ascii=False, indent=2),
        sheet2_json=json.dumps(s2, ensure_ascii=False, indent=2))
    content = [{"type":"input_text","text": preamble + "\n" + user_text},
               {"type":"input_image",
                "image_url":f"data:image/png;base64,{img_b64(pil_img)}","detail":IMAGE_DETAIL}]
    r = client.responses.create(
        model=ARBITER_MODEL, reasoning={"effort":REASONING_ARBITER},
        store=not STATELESS_CALLS,
        input=[{"role":"system","content":[{"type":"input_text","text":ARBITER_SYSTEM}]},
               {"role":"user","content":content}],
        text={"format":{"type":"json_schema","name":"arbiter_reconciliation",
                        "strict":True,"schema":ARBITER_SCHEMA}})
    out = json.loads(r.output_text)
    return out["final"], out["meta"]

print("✅ Cell 8: arbiter ready (GPT-5.5 high-reasoning, soft Claude tiebreaker).")

✅ Cell 8: arbiter ready (GPT-5.5 high-reasoning, soft Claude tiebreaker).


### B7 · Driver

In [22]:
from pathlib import Path

files = sorted([str(p) for p in Path(DRAWING_DIR).iterdir()
                if p.suffix.lower() in (".pdf",".jpg",".jpeg",".png")])
print(f"{len(files)} files → 5.4 ‖ Claude → 5.5 arbiter\n")

rows_s1, rows_s2, rows_s3 = [], [], []
rows_conf = []

import re
_PAGE_RE = re.compile(r"_p(\d+)$")

for i, fp in enumerate(files, 1):
    src = os.path.basename(fp)
    for pidx, page in enumerate(load_pages(fp), 1):
        stem = Path(src).stem
        m = _PAGE_RE.search(stem)
        page_from_name = int(m.group(1)) if m else pidx
        fn = stem if m else (f"{stem}_p{pidx}" if fp.lower().endswith(".pdf") else stem)
        preamble = per_drawing_preamble(fn, src, page_from_name)
        print(f"[{i}/{len(files)}] {fn} ...", end=" ")
        try:
            ...
            # Sheet-1 & Sheet-2 INDEPENDENT (no shared state)
            try: row1 = gpt_extract(page, preamble); err1 = ""
            except Exception as e: row1 = {c:"NULL" for c in DATA_COLS}; err1 = str(e)[:200]
            try: row2 = claude_extract(page, preamble); err2 = ""
            except Exception as e: row2 = {c:"NULL" for c in DATA_COLS}; err2 = str(e)[:200]

            # Sheet-3: arbiter — sees both + image
            try:
                final, meta = arbiter(page, row1, row2, preamble)
                row3 = {c: coerce(final.get(c)) for c in DATA_COLS}
                conf = {c: float(meta.get(c,{}).get("confidence","0") or 0) for c in DATA_COLS}
                chosen = {c: meta.get(c,{}).get("chosen_from","") for c in DATA_COLS}
            except Exception as e:
                # arbiter failure → fall back to whichever extractor succeeded
                fb = row2 if not err2 else (row1 if not err1 else {c:"NULL" for c in DATA_COLS})
                row3 = dict(fb); conf = {c: 0.0 for c in DATA_COLS}; chosen = {c: "fallback" for c in DATA_COLS}

            mean_conf = round(sum(conf.values())/len(conf), 3) if conf else 0.0

            # tag metadata + three-source confidence on Sheet-3
            for r in (row1, row2, row3):
                r["filename"]=fn; r["source_file"]=src; r["source_page"]=pidx
            row3["conf_gpt54_self"]   = row1.get("self_reported_confidence", "NULL")
            row3["conf_claude_self"]  = row2.get("self_reported_confidence", "NULL")
            row3["conf_gpt55_judged"] = str(mean_conf)
            row3["models_ok"]         = ",".join([t for t,e in [("gpt-5.4",err1),("claude",err2)] if not e])

            # confidence row (same shape, for the colouring lookup)
            cr = {c: conf.get(c, 0.0) for c in DATA_COLS}
            cr["filename"]=fn; cr["source_file"]=src; cr["source_page"]=pidx
            cr["conf_gpt54_self"]=row3["conf_gpt54_self"]; cr["conf_claude_self"]=row3["conf_claude_self"]
            cr["conf_gpt55_judged"]=mean_conf

            rows_s1.append(row1); rows_s2.append(row2); rows_s3.append(row3); rows_conf.append(cr)

            with open(os.path.join(JSON_DIR, f"{fn}_pipeline.json"), "w") as f:
                json.dump({"sheet1":row1,"sheet2":row2,"sheet3":row3,
                           "meta":meta if 'meta' in locals() else {},"chosen_from":chosen,
                           "errors":{"sheet1":err1,"sheet2":err2}}, f, indent=2, ensure_ascii=False)
            print(f"✅ ok={row3['models_ok']}  judged_conf={mean_conf}")
        except Exception as e:
            stub = {c:"NULL" for c in COLUMN_ORDER}
            stub["filename"]=fn; stub["source_file"]=src; stub["source_page"]=pidx
            rows_s1.append(dict(stub)); rows_s2.append(dict(stub)); rows_s3.append(dict(stub))
            cr = {c: 0.0 for c in DATA_COLS}; cr["filename"]=fn; cr["source_file"]=src; cr["source_page"]=pidx
            rows_conf.append(cr)
            print(f"❌ {str(e)[:120]}")

14 files → 5.4 ‖ Claude → 5.5 arbiter

[1/14] 404ST6552_DWG_A_page-0001_p1 ... ✅ ok=gpt-5.4,claude  judged_conf=0.921
[2/14] BUSH M2x10.5MM_p1 ... ✅ ok=gpt-5.4,claude  judged_conf=0.944
[3/14] BUSH M2x6MM_page-0001_p1 ... ✅ ok=gpt-5.4,claude  judged_conf=0.949
[4/14] L114A_S7_p1 ... ✅ ok=gpt-5.4,claude  judged_conf=0.949
[5/14] L114A_S7_p2 ... ✅ ok=gpt-5.4,claude  judged_conf=0.954
[6/14] L114B_S3_p1 ... ✅ ok=gpt-5.4,claude  judged_conf=0.921
[7/14] L114B_S3_p2 ... ✅ ok=gpt-5.4,claude  judged_conf=0.936
[8/14] L504_S3 (1)_p1 ... ✅ ok=gpt-5.4,claude  judged_conf=0.96
[9/14] L504_S3 (1)_p2 ... ✅ ok=gpt-5.4,claude  judged_conf=0.951
[10/14] L504_S3 (1)_p3 ... ✅ ok=gpt-5.4,claude  judged_conf=0.962
[11/14] M6 nut-01291_p1 ... ✅ ok=gpt-5.4,claude  judged_conf=0.933
[12/14] M8 nut-101289_p1 ... ✅ ok=gpt-5.4,claude  judged_conf=0.934
[13/14] SP04000476_01-PDF-1-page-001_p1 ... ✅ ok=gpt-5.4,claude  judged_conf=0.951
[14/14] SP04000477_01-PDF-1_p1 ... ✅ ok=gpt-5.4,claude  judged_conf=0.956


### B8 · Write XLSX

In [23]:
import pandas as pd
from openpyxl.styles import PatternFill

SHEET3_EXTRA = ["conf_gpt54_self","conf_claude_self","conf_gpt55_judged","models_ok"]
FULL_S12  = COLUMN_ORDER
FULL_S3   = COLUMN_ORDER + SHEET3_EXTRA
FULL_CONF = COLUMN_ORDER + ["conf_gpt54_self","conf_claude_self","conf_gpt55_judged"]

def frame(rows, cols, fill="NULL"):
    df = pd.DataFrame(rows)
    for c in cols:
        if c not in df.columns: df[c] = fill
    return df[cols]

df_s1   = frame(rows_s1,   FULL_S12).fillna("NULL").replace(r"^\s*$","NULL",regex=True)
df_s2   = frame(rows_s2,   FULL_S12).fillna("NULL").replace(r"^\s*$","NULL",regex=True)
df_s3   = frame(rows_s3,   FULL_S3 ).fillna("NULL").replace(r"^\s*$","NULL",regex=True)
df_conf = frame(rows_conf, FULL_CONF, fill=0.0)

GREEN=PatternFill("solid",fgColor="C6EFCE"); AMBER=PatternFill("solid",fgColor="FFEB9C"); RED=PatternFill("solid",fgColor="FFC7CE")
def fill_for(c):
    try: c=float(c)
    except: return None
    return GREEN if c>=0.85 else AMBER if c>=0.5 else RED
meta_cols = {"filename","source_file","source_page","models_ok",
             "conf_gpt54_self","conf_claude_self","conf_gpt55_judged"}

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    df_s3.to_excel(w, sheet_name="Sheet3_FINAL_arbiter", index=False)   # canonical first
    df_s1.to_excel(w, sheet_name="Sheet1_GPT54",        index=False)
    df_s2.to_excel(w, sheet_name="Sheet2_Claude",        index=False)
    df_conf.to_excel(w, sheet_name="confidence_per_cell", index=False)
    # colour Sheet3 by per-cell judged confidence
    ws = w.sheets["Sheet3_FINAL_arbiter"]; cols=list(df_s3.columns)
    for r in range(len(df_s3)):
        for ci, col in enumerate(cols):
            if col in meta_cols: continue
            f = fill_for(df_conf.iloc[r].get(col, 0.0))
            if f: ws.cell(row=r+2, column=ci+1).fill = f

print(f"✅ XLSX → {OUT_XLSX}")
print(f"   • Sheet3_FINAL_arbiter   ← canonical (color-coded by per-cell confidence)")
print(f"   • Sheet1_GPT54           ← raw output from GPT-5.4")
print(f"   • Sheet2_Claude          ← raw output from Claude")
print(f"   • confidence_per_cell    ← per-cell judged confidence values")
print(f"✅ per-drawing JSON → {JSON_DIR}/*_pipeline.json\n")

# console summary
show=[c for c in df_s3.columns if c not in meta_cols]
for i in range(len(df_s3)):
    print(f"\n{'='*84}\n {df_s3.iloc[i]['filename']} | models_ok={df_s3.iloc[i]['models_ok']} | "
          f"5.4_self={df_s3.iloc[i]['conf_gpt54_self']} claude_self={df_s3.iloc[i]['conf_claude_self']} "
          f"5.5_judged={df_s3.iloc[i]['conf_gpt55_judged']}\n{'='*84}")
    comp = pd.DataFrame({
        "field":   show,
        "5.4":     [str(df_s1.iloc[i][c])[:24] if c in df_s1.columns else "" for c in show],
        "claude":  [str(df_s2.iloc[i][c])[:24] if c in df_s2.columns else "" for c in show],
        "FINAL":   [str(df_s3.iloc[i][c])[:24] for c in show],
        "conf":    [df_conf.iloc[i][c]        for c in show],
    })
    print(comp.to_string(index=False))

✅ XLSX → /content/parallel_arbiter_results.xlsx
   • Sheet3_FINAL_arbiter   ← canonical (color-coded by per-cell confidence)
   • Sheet1_GPT54           ← raw output from GPT-5.4
   • Sheet2_Claude          ← raw output from Claude
   • confidence_per_cell    ← per-cell judged confidence values
✅ per-drawing JSON → /content/audit_json/*_pipeline.json


 404ST6552_DWG_A_page-0001_p1 | models_ok=gpt-5.4,claude | 5.4_self=0.77 claude_self=0.86 5.5_judged=0.921
                          field                       5.4                   claude                     FINAL  conf
                 drawing_number                 404/T6552                404/T6552                 404/T6552  0.99
                       revision                         A                        A                         A  0.97
                   drawing_type       single_part_drawing      single_part_drawing       single_part_drawing  0.95
                    part_family            custom_special           custom_spe

## 4 · Download results

In [24]:
from google.colab import files as colab_files
colab_files.download(OUT_XLSX)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>